# Data Generation

The central challenge of this project is data. Portfolio descriptions supporting both
technology extraction and domain classification don't exist as a labeled corpus anywhere,
so building one is itself a core part of the project.

Two questions define the scope of this notebook:
- **Sourcing**: where do we obtain text matching the target distribution ( project
  descriptions, internship summaries, and extracurricular activity overviews ) ?
- **Labeling**: how do we annotate and filter this text against our target categories (
  technology entities and project domains ) ?

The rest of this notebook documents the pipeline used to answer both, from manual
collection through LLM-assisted augmentation to validation.


## Design Concerns

Before collecting data, three edge cases needed to be represented in the dataset,
a model trained only on "clean" examples won't generalize to real user input.

**Special characters in descriptions.** Users may format descriptions with lists,
bullets, or other special characters.
> Resolved at inference time: descriptions are normalized in preprocessing before
> being passed to the model.

**Descriptions with no technologies mentioned.** A user may still want domain labels
even when their description names no specific technology.
> The model can't invent technologies it was never shown, so the dataset must include
> examples with no technology entities, and the model must be able to output an empty
> entity list rather than hallucinating one. Suggesting technologies based on the
> predicted domain (ranked by confidence) is planned as a downstream post-processing
> step, not something the extraction model itself needs to learn.

**Descriptions spanning multiple domains.** A single project can plausibly belong to
more than one domain (e.g. both "Web Backend" and "DevOps and Cloud Infrastructure").
> The dataset includes multi-label examples, and the classification architecture
> supports multiple, possibly overlapping, domain predictions with independent
> confidence scores.

# 1. Manual Data Collection and Preparation

Starting directly with synthetic data can lead to limited robustness and poor alignment with real-world text distributions. For this reason, we prioritized collecting real-world examples as a first step in the dataset creation process.

## Data Collection Pipeline

The dataset was built using a semi-manual pipeline designed to ensure both quality and relevance of the training data. The process followed these steps:

- Collecting real-world text samples (project descriptions, certifications, and internship experiences) from LinkedIn profiles
- Organizing the extracted information (title, description, and type) into a structured CSV file
- Converting the CSV file into a JSON format using a Python preprocessing script
- Importing the JSON dataset into Label Studio for annotation and classification
- Exporting the annotated data in raw JSONL format, including metadata generated during labeling
- Cleaning and standardizing the exported JSONL using a Python script, keeping only the fields required for model training

This process produced an initial dataset of approximately 60 manually collected and annotated samples, which served as the foundation for the subsequent stages of dataset expansion and model development.

## 2. Data Augmentation Using an LLM

Synthetic data generation offers a solution to this scaling problem. Unlike manual
annotation, LLM-based generation produces diverse, contextually appropriate examples
at a fraction of the time cost — as described by
[Dylan Royan Almeida](https://developers.openai.com/cookbook/examples/sdg1/), who
frames it as a way to substantially extend a dataset's usefulness for downstream tasks.

Using an LLM as a data generator introduces its own risks, which had to be controlled for:
- **Accuracy** — does the generated text make semantic sense for the label it's assigned?
- **Consistency** — do two generations for a similar input produce comparable structure?
- **Diversity** — is the resulting distribution actually more balanced, or does the
  LLM default to a narrow set of patterns?

Generation was done through Claude (Anthropic), distributed across team members to
stay within free-tier rate limits. The prompt below was used to target class balance
based on the current label distribution:

```
I have a JSONL dataset of project descriptions, annotated and classified. Each record has this schema:
json{
  "text": "...",
  "language": "en" or "fr",
  "entities": [{"start": int, "end": int, "label": "TECH", "value": "..."}],
  "domain_labels": ["..."]
}
Entity offsets are critical: start and end must index exactly into text such that text[start:end] == value. No overlapping entities allowed.
Your task: Generate exactly 250 new synthetic samples targeting class balance based on the current distribution in the attached file. Focus generation on the minority classes.
Rules:

All descriptions must be genuinely new — no paraphrasing of existing samples
Mix English and French (~50/50)
Entities must be computed programmatically from the text (use text.find(value)) — never hardcoded
No overlapping entities (if a tech name is a substring of another, keep only the longer/more specific one)
Realistic, domain-accurate descriptions only
Output only the 250 new samples in a .jsonl file — not the originals

Here is my current merged dataset (xxx samples):
```

## 3. Data Validation

Manually collected and synthetically generated samples were merged and passed through
a validation script (`validate_jsonl.py`) enforcing:
- No missing required fields
- Valid entity start/end offsets
- No overlapping entities
- Domain labels restricted to the allowed set
- Domain balance across the merged dataset


In [2]:
# choosing the path and the allowed labels

import json
from pathlib import Path

DATASET_PATH = "../data/raw/dataset_fixed_before_validation.jsonl"

ALLOWED_DOMAIN_LABELS = {
    "Web Frontend",
    "Web Backend",
    "Mobile Development",
    "DevOps and Cloud Infrastructure",
    "Data Engineering",
    "Machine Learning and AI",
    "Cybersecurity",
    "Embedded Systems and IoT",
    "High Performance and Quantum Computing",
}

Here is an example of what our data looks like:

In [3]:
with open(DATASET_PATH,"r",encoding="utf-8") as f:
    first_line = f.readline().strip()

print(first_line)

{"text": "Design and Dev Collab Certificate. Awarded the 'Design and Dev Collab' certificate for participating in a collaborative hackathon focused on bridging UI/UX design and frontend development. Contributed as a Frontend Developer, implementing responsive interfaces using React and modern web technologies while collaborating closely with designers to deliver a functional and user-centered solution under tight deadlines.", "language": "en", "entities": [{"start": 267, "end": 272, "label": "TECH", "value": "React"}], "domain_labels": ["Web Frontend"]}


In [4]:
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    first_line = json.loads(f.readline())

print(first_line)

{'text': "Design and Dev Collab Certificate. Awarded the 'Design and Dev Collab' certificate for participating in a collaborative hackathon focused on bridging UI/UX design and frontend development. Contributed as a Frontend Developer, implementing responsive interfaces using React and modern web technologies while collaborating closely with designers to deliver a functional and user-centered solution under tight deadlines.", 'language': 'en', 'entities': [{'start': 267, 'end': 272, 'label': 'TECH', 'value': 'React'}], 'domain_labels': ['Web Frontend']}


### Illustrating the Validation Logic

The cells below reproduce — in simplified form — the checks performed by the full
validation script (`validate_jsonl.py`). They aren't the production validator itself,
but demonstrate the logic behind each rule.

In [5]:
def validate_existence(sample):
    is_valid = True # if this stays true that means the sample passed all tests
    # Validate if text is a string
    text = sample.get("text")
    if not isinstance(text,str) or not text.strip():
        print(f"This line is not a text or only has spaces")
        is_valid = False

    if(is_valid):
        print("Text is correct!")

validate_existence(first_line)
validate_existence({'text': "  ", 'language': 'en', 'entities': [{'start': 267, 'end': 272, 'label': 'TECH', 'value': 'React'}], 'domain_labels': ['Web Frontend']})

Text is correct!
This line is not a text or only has spaces


In [6]:
def validate_type(sample):
    entities = sample.get("entities",[])
    if not isinstance(entities,list):
        print("entities must be a list")
    else:
        for entity in entities:
            if not isinstance(entity,dict):
                print("entities should contain a list of dicts")
        print("correct")
    
validate_type(first_line)
validate_type({'text': "", 'language': 'en', 'entities': {'start': 267, 'end': 272, 'label': 'TECH', 'value': 'React'}, 'domain_labels': ['Web Frontend']})


correct
entities must be a list


In [7]:
def validate_domain_labels(sample):
    domain_labels = sample.get("domain_labels", [])
    is_valid=True
    if not isinstance(domain_labels,list):
        print("domain labels must be a list")
        is_valid=False
    else:
        for domain in domain_labels:
            if domain not in ALLOWED_DOMAIN_LABELS:
                print(f"invalid domain label :{domain}")
                is_valid = False
    if(is_valid):
        print("correct")

validate_domain_labels(first_line)
validate_domain_labels({'text': "", 'language': 'en', 'entities': {'start': 267, 'end': 272, 'label': 'TECH', 'value': 'React'}, 'domain_labels': ['TEST']})

correct
invalid domain label :TEST


Additional checks are applied to each individual entity:
- Required fields are present and non-empty
- `start` and `end` are integers; `value` is a string
- `start < end`
- `0 <= start` and `end <= len(text)`
- Exact substring match: `text[start:end] == value`
- No duplicate spans
- No overlapping entities

### Validation Results

Running `validate_jsonl.py` on this batch produced:

VALIDATION SUMMARY
Total samples:   247
Valid samples:   101
Invalid samples: 146

146 samples failed validation, almost entirely due to entity offset drift introduced
by the LLM used for the final generation batch. Rather than regenerating these samples
from scratch, the validator's failure report (`validate_output_v1.txt`) was fed back to
Claude alongside the offending file, correcting each flagged entity individually. After
this pass, all 247 samples in this batch passed validation with zero errors.

*This batch (247 samples) reflects an intermediate stage of data collection. The
dataset was later expanded through additional generation rounds to the final
2,350 samples used for training (see `03_Training.ipynb`).*

# 4. Now we will split the dataset into training, validation and test splits

In [8]:
import json
import random
from pathlib import Path

DATASET_PATH = "../data/cleaned/dataset_corrected.jsonl"

TRAIN_RATIO = 0.8
TEST_RATIO = 0.1
VAL_RATIO = 0.1

SEED = 42

OUTPUT_DIR = "splits"

In [9]:
# Data loading
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]

print(f"Loaded : {len(data)} samples")

Loaded : 2350 samples


Inspecting a sample record before and after shuffling confirms the load and shuffle steps behave as expected:

In [10]:
data[0]

{'text': "Design and Dev Collab Certificate. Awarded the 'Design and Dev Collab' certificate for participating in a collaborative hackathon focused on bridging UI/UX design and frontend development. Contributed as a Frontend Developer, implementing responsive interfaces using React and modern web technologies while collaborating closely with designers to deliver a functional and user-centered solution under tight deadlines.",
 'language': 'en',
 'entities': [{'start': 267, 'end': 272, 'label': 'TECH', 'value': 'React'}],
 'domain_labels': ['Web Frontend']}

In [11]:
random.seed(SEED)
random.shuffle(data)

data[0]

{'text': 'Developed a threat hunting platform ingesting Zeek network logs and Windows Event Logs into Splunk. Custom SPL queries and Sigma rules detect lateral movement and privilege escalation patterns.',
 'language': 'en',
 'entities': [{'start': 46, 'end': 50, 'label': 'TECH', 'value': 'Zeek'},
  {'start': 92, 'end': 98, 'label': 'TECH', 'value': 'Splunk'},
  {'start': 123, 'end': 128, 'label': 'TECH', 'value': 'Sigma'}],
 'domain_labels': ['Cybersecurity']}

In [12]:
n = len(data)

train_end = int(n*TRAIN_RATIO)
val_end = train_end + int(n*VAL_RATIO)

train_data = data[:train_end]
val_data = data[train_end:val_end]
test_data = data[val_end:]

print(f"Train : {len(train_data)} samples")
print(f"Validation : {len(val_data)} samples")
print(f"Test : {len(test_data)} samples")

Train : 1880 samples
Validation : 235 samples
Test : 235 samples


In [ ]:
Path(OUTPUT_DIR).mkdir(exist_ok=True)

### Handling Split Imbalance

The random split above doesn't account for label distribution. With only {N} samples
spread across ten domains, a naive random split risks producing a split with zero (or
very few) examples of a minority domain — Cybersecurity being the obvious risk case.

The split computed above is discarded in favor of a **stratified** split, using each
sample's primary domain label as the stratification key via
`sklearn.model_selection.train_test_split`.

In [14]:
from sklearn.model_selection import train_test_split
from collections import Counter

Each sample's primary domain (the first label in domain_labels) is used as the stratification key.

In [15]:
def get_primary_labels(sample):
    domains = sample.get("domain_labels",[])
    if not domains:
        return "NO_LABEL"
    return domains[0]

labels = [get_primary_labels(sample) for sample in data]

print(f"Number of labels stored :{len(labels)}")
Counter(labels)

Number of labels stored :2350


Counter({'Web Backend': 269,
         'Web Frontend': 265,
         'Data Engineering': 257,
         'Mobile Development': 255,
         'Embedded Systems and IoT': 251,
         'High Performance and Quantum Computing': 251,
         'Cybersecurity': 242,
         'DevOps and Cloud Infrastructure': 233,
         'Machine Learning and AI': 226,
         'Other': 101})

In [16]:
train_val, test = train_test_split(
    data,
    test_size=TEST_RATIO,
    random_state=SEED,
    stratify=labels
)

len(train_val)

2115

In [17]:
train_val_labels = [get_primary_labels(sample) for sample in train_val]

val_relative_ratio = VAL_RATIO / (1 - TEST_RATIO)

train, val = train_test_split(
    train_val,
    test_size=val_relative_ratio,
    random_state=SEED,
    stratify=train_val_labels)

print(len(train))
print(len(test))
print(len(val))


1879
235
236


In [54]:
labels_train = [get_primary_labels(sample) for sample in train]

Counter(labels_train)

Counter({'Web Backend': 215,
         'Web Frontend': 211,
         'Data Engineering': 205,
         'Mobile Development': 204,
         'Embedded Systems and IoT': 201,
         'High Performance and Quantum Computing': 201,
         'Cybersecurity': 194,
         'DevOps and Cloud Infrastructure': 187,
         'Machine Learning and AI': 180,
         'Other': 81})

### Summary

The stratified split produced three sets — {train_n} training, {val_n} validation, and
{test_n} test samples — each preserving the overall domain distribution. These splits,
written to `splits/`, are the input to `02_data_preparation.ipynb`, which tokenizes the
text and aligns entity spans to BIO tags at the token level.